# INF-473 - Introducción a la Inteligencia Artificial Explicable
## 1er Semestre 2026 -- Prof. Raquel Pezoa
<center>
    <h1> Enfoque probabilístico no Bayesiano </h2>
</center>

## Bibliotecas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

## Datos con ruido

$$y = ax +b + \epsilon $$

Este ruido es lo que luego modelamos como Gaussiano

In [ ]:
np.random.seed(0)

n = 100
x = np.linspace(0, 10, n)

true_a = 2.0
true_b = 1.0
sigma = 1.5

epsilon = np.random.normal(0, sigma, size=n)

y = true_a * x + true_b + epsilon

## Gráficos
Datos con ruido:
$$y=\mu(x)+\epsilon$$

Acá los puntos son los datos y la recta es el valor real.

In [ ]:
plt.scatter(x, y, label="Datos con ruido")
plt.plot(x, true_a * x + true_b, color='red', label="Media real")
plt.legend()
plt.title(r"$y = \mu(x) + \epsilon$")
plt.show()

## Modelo
Predicción de $\mu(x)$

El modelo solo aprende la media.

Esta función representa la media de la distribución de los datos.

El modelo no predice directamente un valor exacto, sino el centro de una distribución.

In [ ]:
def model(x, a, b):
    return a * x + b

## Negative log-likelihood (NLL)


Definimos la función de pérdida como:

$$\text{NLL} = \sum \log p(y_i | x_i)$$

Esta expresión proviene de asumir que:

$$y_i | x_i ~\sim \text{Normal}(\mu(x_i), \sigma)$$

Interpretación:
- usamos el modelo sobre los datos
- medimos qué tan bien explica cada observación
- penalizamos cuando los datos son poco probables

In [ ]:
def nll(a, b, sigma):
    mu = model(x, a, b)
    return np.sum((y - mu)**2 / (2 * sigma**2) + np.log(sigma))

## Exploración de modelos

Probamos distintos valores de a y b.

Cada par (a, b) define un modelo distinto,
es decir, una distribución distinta para los datos.

Evaluamos cuál de ellos explica mejor los datos.

In [ ]:
a_vals = np.linspace(0, 4, 50)
b_vals = np.linspace(-1, 3, 50)

loss_grid = np.zeros((len(a_vals), len(b_vals)))

for i, a_ in enumerate(a_vals):
    for j, b_ in enumerate(b_vals):
        loss_grid[i, j] = nll(a_, b_, sigma)

## Superficie de la loss

Visualizamos cómo cambia la NLL según los parámetros.

El mínimo corresponde al modelo que mejor explica los datos.

In [ ]:
plt.imshow(loss_grid, extent=[-1,3,0,4], origin='lower', aspect='auto')
plt.colorbar(label="NLL")
plt.xlabel("b")
plt.ylabel("a")
plt.title("Superficie de la loss")
plt.show()

## Mejor modelo

Seleccionamos los parámetros que minimizan la NLL.

Este modelo hace que los datos observados sean lo más probables posible.

In [ ]:
min_idx = np.unravel_index(np.argmin(loss_grid), loss_grid.shape)
best_a = a_vals[min_idx[0]]
best_b = b_vals[min_idx[1]]

print("Best a:", best_a)
print("Best b:", best_b)

## Ajuste del modelo

Mostramos la recta ajustada.

Esta recta corresponde a la función $\mu(x)$ aprendida por el modelo.

In [ ]:
plt.scatter(x, y, label="Datos")
plt.plot(x, model(x, best_a, best_b), color='green', label="Modelo ajustado")
plt.legend()
plt.show()

## Distribución para un punto

Para un valor fijo de x, el modelo define una distribución normal.

Esto significa que el modelo no predice un valor único,
sino una distribución de posibles valores.

In [ ]:
i = 20
x0 = x[i]
mu0 = model(x0, best_a, best_b)

y_range = np.linspace(mu0 - 5, mu0 + 5, 100)
density = norm.pdf(y_range, mu0, sigma)

plt.plot(y_range, density)
plt.axvline(y[i], color='red', linestyle='--', label="y observado")
plt.title(f"Distribución en x = {x0:.2f}")
plt.legend()
plt.show()

## Likelihood

Calculamos $p(y_i | x_i)$

Esto corresponde a usar el modelo en un dato específico,
y medir qué tan bien ese dato coincide con la distribución.

In [ ]:
likelihood_i = norm.pdf(y[i], mu0, sigma)

print("y observado:", y[i])
print("μ(x):", mu0)
print("p(y|x):", likelihood_i)

## Residual (ruido)

El residual:

$$\epsilon  = y - \mu(x)$$

representa el ruido del dato.

La NLL penaliza errores grandes,
ya que estos son menos probables bajo una distribución normal.

In [ ]:
residual = y[i] - mu0

print("ε (ruido):", residual)
print("ε²:", residual**2)

## Distribución del ruido

Visualizamos los residuos.

Esto permite verificar si la suposición de ruido Gaussiano es razonable.

In [ ]:
residuals = y - model(x, best_a, best_b)

plt.hist(residuals, bins=20, density=True)
plt.title("Distribución del ruido ε")
plt.show()

## Interpretación final

Para cada valor de x:
- el modelo define una distribución
- centrada en $\mu(x)$

La recta representa el centro,
y la distribución representa la incertidumbre.

In [ ]:
plt.scatter(x, y, alpha=0.3, label="Datos")

first = True  # para no repetir etiquetas en la leyenda

for xi_idx in range(0, len(x), 15):

    xi = x[xi_idx]
    yi = y[xi_idx]
    mu_i = model(xi, best_a, best_b)

    y_range = np.linspace(mu_i - 4, mu_i + 4, 100)
    density = norm.pdf(y_range, mu_i, sigma)

    density_scaled = density / np.max(density) * 0.5

    # distribución
    plt.plot(xi + density_scaled, y_range, color='gray', alpha=0.5,
             label="Distribución p(y|x)" if first else "")

    # punto observado
    plt.scatter(xi, yi, color='red', s=50,
                label="Dato observado (yᵢ)" if first else "")

    # media
    plt.scatter(xi, mu_i, color='black', s=30,
                label="Media μ(xᵢ)" if first else "")

    first = False

# recta μ(x)
plt.plot(x, model(x, best_a, best_b), color='blue', label="μ(x)")

plt.legend()
plt.title("Cada punto tiene su propia distribución")
plt.xlabel("x")
plt.ylabel("y")
plt.show()